In [1]:
import os
import warnings

os.environ["USER_AGENT"] = "TravelAgent/1.0"
from langchain_core._api import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
BATCH_SIZE = 100  # Safe batch size for Ollama requests
SAVE_DIR = './chroma_travel_db'
EMBEDDING_MODEL = 'nomic-embed-text:latest'
embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)

UK_DESTINATIONS = [
    'Cornwall',
    'North_Cornwall',
    'South_Cornwall',
    'West_Cornwall',
    'Truro_(England)',
    'Newquay',
    'Port_Isaac',
    'St_Ives',
]

In [3]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    urls = [f'https://en.wikivoyage.org/wiki/{destination}' for destination in destinations]
    loader = AsyncHtmlLoader(urls, default_parser="html.parser")
    print("Downloading destination pages ...")
    docs = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
    chunks = sum([splitter.split_documents([d]) for d in docs], [])
    vectordb_client = Chroma.from_documents(
        documents=chunks[:BATCH_SIZE],
        embedding=embedding,
        persist_directory=SAVE_DIR,
    )
    
    for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        print(f"Processing batch {i} to {min(i + BATCH_SIZE, len(chunks))}...")
        vectordb_client.add_documents(batch)
        
    print("Vector store ready.\n")
    return vectordb_client

async def get_travel_info_vectorstore() -> Chroma:
    vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
    return vectorstore_client
    

In [4]:
vectorstore_client = await get_travel_info_vectorstore()

Fetching pages: 100%|#######################################################################################################| 8/8 [00:00<00:00, 11.98it/s]


Processing batch 100 to 200...
Processing batch 200 to 300...
Processing batch 300 to 400...
Processing batch 400 to 500...
Processing batch 500 to 600...
Processing batch 600 to 700...
Processing batch 700 to 800...
Processing batch 800 to 900...
Processing batch 900 to 1000...
Processing batch 1000 to 1100...
Processing batch 1100 to 1200...
Processing batch 1200 to 1300...
Processing batch 1300 to 1400...
Processing batch 1400 to 1500...
Processing batch 1500 to 1600...
Processing batch 1600 to 1700...
Processing batch 1700 to 1800...
Processing batch 1800 to 1900...
Processing batch 1900 to 2000...
Processing batch 2000 to 2100...
Processing batch 2100 to 2200...
Processing batch 2200 to 2300...
Processing batch 2300 to 2339...
Vector store ready.

